In [ ]:
import pandas as pd


df_merged = pd.read_csv("../data/processed/raw_merged_dataset.csv")
df_raw = pd.read_csv("../data/raw/p01/pmsys/wellness.csv")


print("--- DONNÉES BRUTES (Participant 01) ---")
display(df_raw[['effective_time_frame', 'fatigue']].head(2))


print("\n--- NOTRE DATASET FUSIONNÉ ---")
display(df_merged[['participant_id', 'date', 'fatigue']].head(2))

--- DONNÉES BRUTES (Participant 01) ---


,effective_time_frame,fatigue
0,2019-11-01T08:31:40.751Z,2
1,2019-11-02T10:00:01.229Z,2



--- NOTRE DATASET FUSIONNÉ ---


,participant_id,date,fatigue
0,1,2019-11-01,2.0
1,1,2019-11-02,2.0


In [ ]:
import json
import pandas as pd

df_merged = pd.read_csv("../data/processed/raw_merged_dataset.csv")

print("--- 1. TEST FITBIT STEPS (Participant p03) ---")
try:
    with open("../data/raw/p03/fitbit/steps.json", 'r') as f:
        raw_steps = json.load(f)
        
    df_raw_steps = pd.DataFrame(raw_steps)
    df_raw_steps['date'] = pd.to_datetime(df_raw_steps['dateTime']).dt.date
    df_raw_steps['value'] = pd.to_numeric(df_raw_steps['value'])
    
    test_date = df_raw_steps['date'].iloc[0]
    expected_steps = df_raw_steps[df_raw_steps['date'] == test_date]['value'].sum()
    print(f"Brut -> Somme des pas calculée pour le {test_date} (p03) : {expected_steps}")
 
    p3_mask = df_merged['participant_id'].astype(str).str.contains('3')
    date_mask = df_merged['date'].astype(str) == str(test_date)
    test_row = df_merged[p3_mask & date_mask]
    
    if not test_row.empty:
        merged_steps = test_row['total_steps'].values[0]
        print(f"Agrégé -> Pas enregistrés dans la table finale : {merged_steps}")
        if expected_steps == merged_steps:
            print("✅ SUCCÈS : Les données JSON de pas sont parfaitement agrégées !")
        else:
            print(f"❌ DÉCALAGE : Brut = {expected_steps} vs Agrégé = {merged_steps}")
    else:
        print("❌ Ligne non trouvée dans le dataset pour p03 à cette date.")
except Exception as e:
    print(f"Erreur durant le test : {e}")


print("\n--- 2. TEST VARIABLE CIBLE (is_injured) ---")
total_days = len(df_merged)
total_injuries = df_merged['is_injured'].sum()

print(f"Total de jours enregistrés : {total_days}")
print(f"Nombre de jours avec blessure signalée (is_injured == 1) : {int(total_injuries)}")

if total_injuries > 0:
    print("✅ SUCCÈS : La table contient des événements de blessure.")
    display(df_merged[df_merged['is_injured'] == 1][['participant_id', 'date', 'is_injured']].head(5))
else:
    print("⚠️ ATTENTION : 0 blessure détectée. Vérifie le parsing du fichier injury.csv.")

--- 1. TEST FITBIT STEPS (Participant p03) ---
Brut -> Somme des pas calculée pour le 2019-11-01 (p03) : 11204
Agrégé -> Pas enregistrés dans la table finale : 11204.0
✅ SUCCÈS : Les données JSON de pas sont parfaitement agrégées !

--- 2. TEST VARIABLE CIBLE (is_injured) ---
Total de jours enregistrés : 456
Nombre de jours avec blessure signalée (is_injured == 1) : 45
✅ SUCCÈS : La table contient des événements de blessure.


,participant_id,date,is_injured
6,1,2019-11-07,1.0
10,1,2019-11-11,1.0
17,1,2019-11-18,1.0
24,1,2019-11-25,1.0
31,1,2019-12-02,1.0


In [ ]:
import pandas as pd

df_merged = pd.read_csv("../data/processed/raw_merged_dataset.csv")

print("--- 3. TEST DE COUVERTURE TEMPORELLE & COMPLÉTUDE ---")


participants = sorted(df_merged['participant_id'].astype(str).unique())
print(f"Participants identifiés dans la base : {participants}")

print("\n--- Répartition par participant ---")
for p in participants:
    df_p = df_merged[df_merged['participant_id'].astype(str) == p]
    min_date = df_p['date'].min()
    max_date = df_p['date'].max()
    nb_jours = len(df_p)
    print(f"Participant {p} : {nb_jours} jours (du {min_date} au {max_date})")


print("\n--- BILAN DES VALEURS MANQUANTES (%) ---")
missing_summary = pd.DataFrame({
    'Valeurs présentes': df_merged.notnull().sum(),
    'Valeurs manquantes (NaN)': df_merged.isnull().sum(),
    '% Manquant': (df_merged.isnull().mean() * 100).round(2)
}).sort_values(by='% Manquant', ascending=False)

display(missing_summary)

--- 3. TEST DE COUVERTURE TEMPORELLE & COMPLÉTUDE ---
Participants identifiés dans la base : ['1', '3', '5']

--- Répartition par participant ---
Participant 1 : 152 jours (du 2019-11-01 au 2020-03-31)
Participant 3 : 152 jours (du 2019-11-01 au 2020-03-31)
Participant 5 : 152 jours (du 2019-11-01 au 2020-03-31)

--- BILAN DES VALEURS MANQUANTES (%) ---


,Valeurs présentes,Valeurs manquantes (NaN),% Manquant
avg_heart_rate,0,456,100.00
perceived_exertion,41,415,91.01
duration_min,41,415,91.01
calories_consumed,158,298,65.35
composition_score,340,116,25.44
overall_score,340,116,25.44
sleep_log_entry_id,340,116,25.44
deep_sleep_in_minutes,340,116,25.44
resting_heart_rate,340,116,25.44
revitalization_score,340,116,25.44
